# Fragrantica Exploration and Cleaning

## Objective
Explore, assess, and clean the Fragrantica reference dataset obtained from Kaggle to prepare it for downstream fragrance analytics and marketplace product matching.

## Source Data
- **Source:** Kaggle Fragrantica dataset
- **Format:** CSV
- **Location:** `/Volumes/main/fragrance_analytics/reference_data/fra_perfumes.csv`
- **Working DataFrame:** `df_perfumes`

The original CSV file remains unchanged. All transformations are applied to the working DataFrame, and the processed dataset is stored separately.

## Workflow

### 1. Exploratory Data Analysis
- Inspect the schema, sample records, row counts, missing values, and column distributions.
- Investigate exact duplicate records and repeated fragrance URLs.
- Identify data quality issues requiring cleaning.

### 2. Data Cleaning and Standardization
- Standardize column names to lowercase `snake_case`.
- Standardize gender categories: `for women` → `women`, `for men` → `men`, and `for women and men` → `unisex`.
- Convert `rating_value` to `double`.
- Convert `rating_count` to `long`, removing thousands separators and handling missing or invalid values.
- Remove exact duplicate rows.
- Resolve duplicate URLs by retaining the record with the highest `rating_count`; records with missing rating counts are ranked last.
- Generate a deterministic `id` from the fragrance URL using SHA-256.
- Validate the cleaned schema, identifiers, gender categories, URL uniqueness, and final record counts.

### 3. Data Storage
Save the cleaned dataset as a Delta table, keeping the original CSV source unchanged.

## Output
**Delta table:** `main.fragrance_analytics.fragrantica_perfumes`

**Final schema:**
- `id` — string, deterministic identifier derived from the fragrance URL
- `name` — string
- `gender` — string, standardized category (`women`, `men`, or `unisex`)
- `rating_value` — double
- `rating_count` — long
- `main_accords` — string
- `perfumers` — string
- `description` — string
- `url` — string

## Technical Stack
Databricks, Python, PySpark, Delta Lake.

## Downstream Use
The cleaned reference dataset supports fragrance accord analysis and matching relevant Fragrantica products to cleaned marketplace listings.

## Data Integrity Note
The source CSV remains immutable. Exact duplicate rows are removed first, followed by URL-based deduplication that prioritizes the record with the highest available rating count. Deterministic identifiers are derived from the URL to remain consistent across notebook reruns. Gender labels are standardized in the cleaned dataset to support consistent grouping, analysis, and visualization.

In [0]:
# Inspect fra_perfumes.csv — header, schema, dimensions, and samples

csv_path_perfumes = "/Volumes/main/fragrance_analytics/reference_data/fra_perfumes.csv"

df_perfumes = spark.read.csv(csv_path_perfumes, header=True, inferSchema=True)

# Header (column names)
print("Header:")
print(df_perfumes.columns)

# Schema
print("\nSchema:")
df_perfumes.printSchema()

# Dimensions
print(f"Number of rows:    {df_perfumes.count():,}")
print(f"Number of columns: {len(df_perfumes.columns):,}")

# First 20 rows
print("\nFirst 20 rows:")
display(df_perfumes.limit(20))

In [0]:

from pyspark.sql import functions as F

# 1. Check missing values in each column
missing_counts = df_perfumes.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            c
        )
    ).alias(c)
    for c in df_perfumes.columns
])

print("Missing values by column:")
display(missing_counts)

# 2. Inspect gender categories
print("Gender distribution:")
display(
    df_perfumes
    .groupBy("Gender")
    .count()
    .orderBy(F.desc("count"))
)

# 3. Check exact duplicate rows
total_rows = df_perfumes.count()
distinct_rows = df_perfumes.distinct().count()

print(f"Total rows:           {total_rows:,}")
print(f"Distinct rows:        {distinct_rows:,}")
print(f"Duplicate rows:       {total_rows - distinct_rows:,}")

# 4. Check duplicate URLs
url_stats = df_perfumes.agg(
    F.count("url").alias("non_null_urls"),
    F.countDistinct("url").alias("distinct_urls")
)

print("URL statistics:")
display(url_stats)

# 5. Inspect the Main Accords format
print("Sample Main Accords values:")
display(
    df_perfumes
    .select("Name", "Gender", "Main Accords")
    .limit(15)
)

In [0]:
# Standardize and inspect gender categories
df_perfumes = df_perfumes.withColumn(
    "gender",
    F.when(F.lower(F.trim(F.col("gender"))) == "for women", "women")
     .when(F.lower(F.trim(F.col("gender"))) == "for men", "men")
     .when(F.lower(F.trim(F.col("gender"))) == "for women and men", "unisex")
     .otherwise(F.lower(F.trim(F.col("gender"))))
)

display(
    df_perfumes
    .groupBy("gender")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

source_columns = df_perfumes.columns

# 1. Find exact duplicate records across all source columns
duplicate_groups = (
    df_perfumes
    .groupBy(*source_columns)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print("Groups of exact duplicate records:")
display(duplicate_groups)

# 2. Show all rows belonging to exact duplicate groups, including those containing NULL values

source = df_perfumes.alias("src")
duplicates = duplicate_groups.select(*source_columns).alias("dup")

join_condition = F.lit(True)

for c in source_columns:
    join_condition = join_condition & (
        F.col(f"src.`{c}`").eqNullSafe(F.col(f"dup.`{c}`"))
    )

exact_duplicates = (
    source
    .join(duplicates, on=join_condition, how="inner")
    .select("src.*")
)

print("All rows in exact duplicate groups, including NULLs:")
display(exact_duplicates.orderBy("url"))

# 3. Inspect URLs associated with multiple records
duplicate_urls = (
    df_perfumes
    .groupBy("url")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("Name").alias("distinct_names"),
        F.countDistinct("Gender").alias("distinct_genders")
    )
    .filter(F.col("row_count") > 1)
    .orderBy(F.desc("row_count"))
)

print("URLs appearing more than once:")
display(duplicate_urls)


In [0]:
# Identify URLs associated with multiple non-identical records
duplicate_urls = (
    df_perfumes
    .groupBy("url")
    .agg(F.count("*").alias("records_per_url"))
    .filter(F.col("records_per_url") > 1)
)

# Retrieve all records associated with these URLs
df_duplicate_url_rows = (
    df_perfumes
    .join(
        duplicate_urls.select("url", "records_per_url"),
        on="url",
        how="inner"
    )
    .orderBy("url")
)

# Summarize duplicate URL groups
print(f"Duplicate URL groups: {duplicate_urls.count()}")
print(f"Total rows in these groups: {df_duplicate_url_rows.count()}")

display(df_duplicate_url_rows)

In [0]:

from pyspark.sql import functions as F

# 1. Inspect distinct values in Rating Value
print("Distinct Rating Value entries:")
display(
    df_perfumes
    .groupBy("Rating Value")
    .count()
    .orderBy("Rating Value")
)

# 2. Inspect distinct values in Rating Count
print("Distinct Rating Count entries:")
display(
    df_perfumes
    .groupBy("Rating Count")
    .count()
    .orderBy("Rating Count")
)


In [0]:

from pyspark.sql import functions as F

# 1. Rename all columns to lowercase snake_case
df_perfumes = df_perfumes.toDF(
    "name",
    "gender",
    "rating_value",
    "rating_count",
    "main_accords",
    "perfumers",
    "description",
    "url"
)

# 2. Clean rating_value and convert to double
df_perfumes = df_perfumes.withColumn(
    "rating_value",
    F.when(
        F.trim(F.col("rating_value")).isin(
            "", "N/A", "n/a", "null"
        ),
        F.lit(None).cast("double")
    ).otherwise(
        F.trim(F.col("rating_value")).cast("double")
    )
)

# 3. Clean rating_count and convert to long
df_perfumes = df_perfumes.withColumn(
    "rating_count",
    F.when(
        F.trim(F.col("rating_count")).isin(
            "", "N/A", "n/a", "null"
        ),
        F.lit(None).cast("long")
    ).otherwise(
        F.regexp_replace(
            F.trim(F.col("rating_count")),
            ",",
            ""
        ).cast("long")
    )
)

df_perfumes.printSchema()

display(df_perfumes.limit(10))
print("Total rows:", df_perfumes.count())
print("Distinct URLs:", df_perfumes.select("url").distinct().count())
print("Exact duplicate rows:", df_perfumes.count() - df_perfumes.dropDuplicates().count())

In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window

# 1. Remove exact duplicate rows
df_perfumes = df_perfumes.dropDuplicates()

# 2. Keep the record with the highest rating count for each URL
url_window = Window.partitionBy("url").orderBy(
    F.col("rating_count").desc_nulls_last()
)

df_perfumes = (
    df_perfumes
    .withColumn("url_rank", F.row_number().over(url_window))
    .filter(F.col("url_rank") == 1)
    .drop("url_rank")
)

# 3. Generate a deterministic ID from the URL
df_perfumes = df_perfumes.withColumn(
    "id",
    F.sha2(F.col("url"), 256)
)

# 4. Place ID as the first column
df_perfumes = df_perfumes.select(
    "id",
    "name",
    "gender",
    "rating_value",
    "rating_count",
    "main_accords",
    "perfumers",
    "description",
    "url"
)

# 5. Validate the final DataFrame
print("Final row count:", df_perfumes.count())
print("Distinct URL count:", df_perfumes.select("url").distinct().count())
print("Distinct ID count:", df_perfumes.select("id").distinct().count())
print("Null IDs:", df_perfumes.filter(F.col("id").isNull()).count())

df_perfumes.printSchema()

# 6. Replace the existing Delta table with the updated data
(
    df_perfumes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.fragrantica_perfumes")
)

print("Delta table recreated with the updated data.")